# Semana 7 — El Algoritmo de Grover
## Búsqueda cuántica: de O(N) a O(√N)

**Objetivo:** Implementar el algoritmo de Grover desde cero, entender el operador de difusión y visualizar cómo las amplitudes rotan hacia la solución en cada iteración.

**Hito verificable:** Implementas Grover para n=3 qubits (8 elementos), encuentras el estado marcado con alta probabilidad en ≈2 iteraciones y graficas la evolución de las amplitudes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
print('Librerías importadas correctamente')

## Ejercicio 7.1 — El problema de búsqueda no estructurada
Tenemos N elementos. Uno (o varios) están marcados. Clásico: O(N). Grover: O(√N).

In [ ]:
# Comparación de complejidades
tamanos = [4, 8, 16, 64, 256, 1024, 1_000_000]

print(f'{"N elementos":>15}  {"Clásico O(N)":>15}  {"Grover O(√N)":>15}  {"Aceleración":>12}')
print('-' * 65)
for N in tamanos:
    clasico = N // 2  # promedio
    grover = int(np.pi / 4 * np.sqrt(N))
    aceleracion = clasico / max(grover, 1)
    print(f'{N:>15,}  {clasico:>15,}  {grover:>15,}  {aceleracion:>12.1f}x')

print()
print('El número óptimo de iteraciones de Grover es π/4 · √N')

## Ejercicio 7.2 — Los dos operadores de Grover
Grover combina: (1) Oráculo: marca el estado buscado con fase -1. (2) Difusor: amplifica el estado marcado.

In [ ]:
def oraculo(n_qubits, objetivo):
    """Construye el oráculo que marca el estado |objetivo⟩ con fase -1."""
    N = 2**n_qubits
    O = np.eye(N, dtype=complex)
    O[objetivo, objetivo] = -1  # Solo se invierte la fase del estado objetivo
    return O

def difusor(n_qubits):
    """Operador de difusión de Grover: reflexión alrededor de la media."""
    N = 2**n_qubits
    # D = 2|s⟩⟨s| - I, donde |s⟩ = superposición uniforme
    s = np.ones(N, dtype=complex) / np.sqrt(N)
    D = 2 * np.outer(s, s.conj()) - np.eye(N, dtype=complex)
    return D

# Verificar que son unitarios
n = 3
O = oraculo(n, objetivo=5)  # Buscamos el estado |101⟩ = 5
D = difusor(n)

print(f'Oráculo unitario: {np.allclose(O.conj().T @ O, np.eye(2**n))}')
print(f'Difusor unitario: {np.allclose(D.conj().T @ D, np.eye(2**n))}')
print()
print('Oráculo: diagonal con -1 en posición 5 (estado |101⟩):')
print(np.round(np.diag(O), 0))

## Ejercicio 7.3 — Grover paso a paso: visualizar la rotación de amplitudes

In [ ]:
def grover_iteraciones(n_qubits, objetivo, max_iter=None):
    """Ejecuta el algoritmo de Grover y devuelve el historial de amplitudes."""
    N = 2**n_qubits
    if max_iter is None:
        max_iter = int(np.pi / 4 * np.sqrt(N)) + 1
    
    O = oraculo(n_qubits, objetivo)
    D = difusor(n_qubits)
    G = D @ O  # Un paso de Grover
    
    # Estado inicial: superposición uniforme
    psi = np.ones(N, dtype=complex) / np.sqrt(N)
    historial = [psi.copy()]
    
    for _ in range(max_iter):
        psi = G @ psi
        historial.append(psi.copy())
    
    return historial

n = 3  # 8 elementos
objetivo = 5  # Estado |101⟩
historial = grover_iteraciones(n, objetivo, max_iter=4)

fig, axes = plt.subplots(1, len(historial), figsize=(14, 3))
for i, (psi, ax) in enumerate(zip(historial, axes)):
    probs = np.abs(psi)**2
    colores = ['red' if j == objetivo else 'steelblue' for j in range(2**n)]
    ax.bar(range(2**n), probs, color=colores)
    ax.set_ylim(0, 1)
    ax.set_xticks(range(2**n))
    ax.set_xticklabels([f'|{j:03b}⟩' for j in range(2**n)], rotation=90, fontsize=7)
    titulo = 'Inicio' if i == 0 else f'Iter {i}'
    ax.set_title(f'{titulo}\nP(objetivo)={probs[objetivo]:.3f}')
    ax.axhline(1/2**n, color='gray', linestyle='--', alpha=0.5)

plt.suptitle(f'Algoritmo de Grover: n={n} qubits, buscando |{objetivo:03b}⟩ (rojo)', y=1.02)
plt.tight_layout()
plt.savefig('grover_amplitudes.png', dpi=100, bbox_inches='tight')
plt.show()

print('Iteración 1 ya da muy alta probabilidad. El pico rojo crece con cada paso.')

## Ejercicio 7.4 — Número óptimo de iteraciones

In [ ]:
# P(éxito) = sin²((2k+1)θ) donde sin(θ) = 1/√N
# El máximo se alcanza en k ≈ π/4 · √N iteraciones

n = 4  # 16 elementos
N = 2**n
objetivo = 10
max_iter = 20

historial = grover_iteraciones(n, objetivo, max_iter=max_iter)
p_objetivo = [np.abs(psi[objetivo])**2 for psi in historial]

k_optimo = int(np.pi / 4 * np.sqrt(N))
theta = np.arcsin(1/np.sqrt(N))
iters = np.arange(max_iter + 1)
p_teorica = np.sin((2*iters + 1) * theta)**2

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(iters, p_objetivo, 'bo-', label='P(éxito) simulada', linewidth=2, markersize=6)
ax.plot(iters, p_teorica, 'r--', label='P(éxito) teórica', linewidth=1.5)
ax.axvline(k_optimo, color='green', linestyle=':', label=f'k óptimo = {k_optimo}')
ax.set_xlabel('Número de iteraciones k')
ax.set_ylabel('P(medir el estado objetivo)')
ax.set_title(f'Grover: n={n} qubits (N={N}), k_óptimo = ⌊π/4·√N⌋ = {k_optimo}')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('grover_iteraciones.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'N={N}: k_óptimo = {k_optimo}, P(éxito) máxima = {max(p_objetivo):.4f}')
print('Si se hacen demasiadas iteraciones, la probabilidad vuelve a caer → NO más es mejor.')

## Ejercicio 7.5 — Grover en Qiskit (circuito real)

In [ ]:
# Grover para n=2 qubits, buscando |11⟩ = 3
# Oráculo para |11⟩: CZ (fase -1 solo cuando ambos son 1)

def grover_2qubits_qiskit():
    qc = QuantumCircuit(2, 2)
    
    # Inicialización: superposición uniforme
    qc.h([0, 1])
    qc.barrier()
    
    # Una iteración de Grover para |11⟩
    # Oráculo: CZ marca |11⟩ con fase -1
    qc.cz(0, 1)
    qc.barrier()
    
    # Difusor: 2|s⟩⟨s| - I = H(2|0⟩⟨0| - I)H
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    qc.barrier()
    
    qc.measure([0, 1], [0, 1])
    return qc

qc_grover = grover_2qubits_qiskit()
print('Circuito de Grover (n=2, buscando |11⟩):')
print(qc_grover.draw(output='text'))

sim = AerSimulator()
job = sim.run(qc_grover, shots=10000)
counts = job.result().get_counts()
print(f'\nResultados: {counts}')
print(f'P(|11⟩) = {counts.get("11", 0)/10000:.4f}  (esperado: 1.0 para N=4, k=1)')

## Mini reto — Completar antes de avanzar a la Semana 8

Generaliza el circuito de Grover para n=3 qubits en Qiskit con oráculo configurable:
1. El oráculo debe poder marcar cualquier estado de 3 bits
2. Usa la construcción con puertas CNOT multi-control (MCT en Qiskit)
3. Ejecuta con k=2 iteraciones y verifica P(objetivo) > 0.95
4. Compara los resultados del simulador vs. la fórmula teórica sin²((2k+1)θ)

In [ ]:
# TU CÓDIGO AQUÍ
def grover_3qubits(objetivo: int, n_iters: int = 2) -> QuantumCircuit:
    """Circuito de Grover para n=3 qubits con oráculo configurable."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 7

Antes de avanzar, comprueba que puedes:

- [ ] Implementar el oráculo y el difusor de Grover como matrices
- [ ] Calcular el número óptimo de iteraciones para un N dado
- [ ] Visualizar la evolución de las amplitudes en cada iteración
- [ ] Implementar Grover de 2 qubits en Qiskit
- [ ] Verificar que demasiadas iteraciones degradan el resultado

## Reflexión (escribe aquí tu respuesta)

**¿Por qué Grover no puede acelerar exponencialmente (como Shor), sino solo cuadráticamente?**

*(tu respuesta aquí)*

**¿Qué significa el operador de difusión geométricamente en la esfera de Bloch (espacio de 2 dimensiones del subespacio de Grover)?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 8 — Transformada Cuántica de Fourier*